# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` and other required libraries are installed
!pip install mlcroissant --quiet
!pip install pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}\nVersion: {metadata.version}")

## 2. Data Overview
Explore available record sets, their fields and IDs. All entities in Croissant, including record sets, fields, and columns, are referenced by their `@id` fields (URIs or unique string IDs).

In [ ]:
# List all record sets defined in the dataset schema using their @id
record_sets = list(dataset.record_sets)

print("Available record sets by @id:")
for rset in record_sets:
    print(f"- {rset['@id']}: {rset.get('name', '')}")

# For illustration, print out the fields for each record set, by @id
print("\nRecord set fields by @id:")
for rset in record_sets:
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nRecord set {rset['@id']} fields:")
    for field in fields:
        field_id = field.get('@id', 'N/A')
        fname = field.get('name', '')
        dtype = field.get('dataType', '')
        print(f"  - {field_id} (name: {fname}, type: {dtype})")

## 3. Data Extraction
Load data from each record set into a DataFrame for further processing. Reference record sets and field names by their `@id`.

In [ ]:
# List of record set @ids (from the previous overview step, replace with actual @ids)
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

# For demonstration, load all available record sets
for rset_id in record_set_ids:
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rset_id)))
        if not df.empty:
            dataframes[rset_id] = df
            print(f"Loaded record set: {rset_id} (columns: {df.columns.tolist()})")
    except Exception as e:
        print(f"Could not load records for {rset_id}: {e}")

# Pick the main record set for downstream analysis (use the largest populated one)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nMain record set selected: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No records were found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filter records, normalize numeric fields, group data by attributes. All column/field references must use their `@id`.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    print(f"Columns in the main record set (use @id as column references):\n{df.columns.tolist()}")

    # Try to identify a numeric column for demonstration (fall back to 'Age')
    numeric_col_id = None
    for c in df.columns:
        if 'Age' in c or 'age' in c or df[c].dtype in ('int64', 'float64'):
            numeric_col_id = c
            break
    if numeric_col_id is None:
        numeric_col_id = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else df.columns[0]
    print(f"\nUsing numeric field for analysis: {numeric_col_id}")

    # Filter: keep records with value > threshold
    threshold = df[numeric_col_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_col_id]) else 0
    if pd.api.types.is_numeric_dtype(df[numeric_col_id]):
        filtered_df = df[df[numeric_col_id] > threshold]
        print(f"Filtered records with {numeric_col_id} > {threshold:.2f} (showing up to 5):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_col_id}_normalized"] = (filtered_df[numeric_col_id] - filtered_df[numeric_col_id].mean()) / filtered_df[numeric_col_id].std()
        print(f"\nNormalized {numeric_col_id} for filtered records (showing up to 5):")
        print(filtered_df[[numeric_col_id, f"{numeric_col_id}_normalized"]].head())
    else:
        print(f"Column {numeric_col_id} is not numeric.")

    # Try grouping by a categorical field (e.g. 'Sex' or 'Anatomical location')
    possible_group_fields = [c for c in df.columns if 'sex' in c.lower() or 'location' in c.lower() or 'MSI' in c or 'site' in c]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field and pd.api.types.is_numeric_dtype(df[numeric_col_id]):
        grouped_df = filtered_df.groupby(group_field)[numeric_col_id].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_col_id}):")
        print(grouped_df.head())
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize distribution or relationships. Column names are `@id`s from schema.

In [ ]:
if main_record_set_id and pd.api.types.is_numeric_dtype(df[numeric_col_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col_id].dropna(), kde=True, bins=10, color='royalblue')
    plt.title(f"Distribution of {numeric_col_id}")
    plt.xlabel(numeric_col_id)
    plt.ylabel("Count")
    plt.show()

    # If categorical group field found, boxenplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxenplot(x=group_field, y=numeric_col_id, data=df, palette='Set2')
        plt.title(f"{numeric_col_id} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion

- Explored clinical and pathological variables of second primary colorectal cancer survivors, loaded via the Croissant schema.
- Applied filtering and normalization using column `@id`s for robust, reproducible data processing.
- Grouped and visualized numeric data by categorical clinical attributes.

Refer to [FAIR^2 Project on SenScience](https://sen.science/doi/10.71728/senscience.qs2f-h81p) for full context and citation.

This notebook template can be readily adapted for broader Croissant-based FAIR dataset exploration with `mlcroissant`.